In [ ]:
%pip install -r requirements.txt

In [ ]:
from dotenv import load_dotenv

load_dotenv()

Import Statements


In [ ]:
import os
import re
import json
from typing import List, Dict, Any, Optional, Tuple
from enum import Enum
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from langchain_openai import ChatOpenAI, OpenAIEmbeddings



import ipywidgets as widgets
from IPython.display import display, clear_output


from langsmith import Client
from langchain_core.tracers.context import tracing_v2_enabled

#LangSmith
client = Client()



##Set up LLM and Stopwords

In [ ]:
USE_OPENROUTER = True  # Set to True to use OpenRouter, False to use OpenAI API

if USE_OPENROUTER:
    LLM_MODEL = "openai/gpt-5-mini"
    API_BASE = "https://openrouter.ai/api/v1"
else:
    LLM_MODEL = "gpt-4o-mini"
    API_BASE = "https://api.openai.com/v1"  

EMBEDDING_MODEL = "text-embedding-3-small"

SIMILARITY_THRESHOLD = 0.6
N_RETRIEVAL_RESULTS = 1

stop_words = set(stopwords.words('english'))
print(f"Loaded {len(stop_words)} stopwords")

In [ ]:
def get_llm():

    llm = ChatOpenAI(
        model=LLM_MODEL,
        openai_api_base=API_BASE if USE_OPENROUTER else None,
        openai_api_key=os.environ["OPENAI_API_KEY"],
        temperature=0,
        max_tokens=2000,
        default_headers={
            "HTTP-Referer": "https://medicare-chatbot.local",
            "X-Title": "MediCare Chatbot"
        }
    )
    return llm

def get_embeddings():

    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        openai_api_base="https://openrouter.ai/api/v1",
        openai_api_key=os.environ["OPENAI_API_KEY"]
    )
    return embeddings

llm = get_llm()
embeddings = get_embeddings()

### Patient Database

In [ ]:
PATIENTS_DB = { 
    "P001": { 
        "name": "María García", 
        "email": "maria.garcia@email.com", 
        "phone": "+34 612 345 678", 
        "date_of_birth": "1985-03-15", 
        "membership_type": "Premium", 
        "balance_due": 45.00, 
        "upcoming_appointments": [ 
            { 
                "date": "2024-02-15", 
                "time": "10:30", 
                "doctor": "Dr. Fernández", 
                "specialty": "General Medicine", 
                "status": "confirmed" 
            }, 
            { 
                "date": "2024-03-01", 
                "time": "16:00", 
                "doctor": "Dr. López", 
                "specialty": "Dermatology", 
                "status": "pending_confirmation" 
            } 
        ], 
        "last_visit": "2024-01-20", 
        "primary_doctor": "Dr. Fernández", 
        "allergies": ["Penicillin"], 
        "insurance_provider": "Sanitas" 
    }, 
    "P002": { 
        "name": "Carlos Rodríguez", 
        "email": "carlos.r@email.com", 
        "phone": "+34 698 765 432", 
        "date_of_birth": "1972-11-28", 
        "membership_type": "Basic", 
        "balance_due": 0.00, 
        "upcoming_appointments": [], 
        "last_visit": "2023-11-05", 
        "primary_doctor": "Dr. Martín", 
        "allergies": [], 
        "insurance_provider": "Adeslas" 
    }, 
    "P003": { 
        "name": "Ana Belén Torres", 
        "email": "anabelen.t@email.com", 
        "phone": "+34 655 123 789", 
        "date_of_birth": "1990-07-22", 
        "membership_type": "Premium", 
        "balance_due": 120.50, 
        "upcoming_appointments": [ 
            { 
                "date": "2024-02-10", 
                "time": "09:00", 
                "doctor": "Dr. Sánchez", 
                "specialty": "Gynecology", 
                "status": "confirmed" 
            } 
        ], 
        "last_visit": "2024-01-28", 
        "primary_doctor": "Dr. Sánchez", 
        "allergies": ["Ibuprofen", "Latex"], 
        "insurance_provider": "Mapfre" 
    } 
} 

### Clinic FAQs

In [ ]:
CLINIC_FAQS = { 
    "FAQ001": { 
        "category": "Appointments", 
        "question": "How can I schedule an appointment?", 
        "answer": "You can schedule an appointment through our online portal at medicare-plus.com/appointments, by calling our reception at +34 911 234 567 (Monday to Friday, 8:00-20:00), or through this chat by requesting to speak with our scheduling team." 
    }, 
    "FAQ002": { 
        "category": "Appointments", 
        "question": "What is the cancellation policy?", 
        "answer": "Appointments must be cancelled at least 24 hours in advance to avoid a €20 cancellation fee. Premium members can cancel up to 12 hours before without penalty. To cancel, use our online portal or call reception." 
    }, 
    "FAQ003": { 
        "category": "Appointments", 
        "question": "How early should I arrive for my appointment?", 
        "answer": "Please arrive 15 minutes before your scheduled appointment time. First-time patients should arrive 30 minutes early to complete registration paperwork." 
    }, 
    "FAQ004": { 
        "category": "Payments", 
        "question": "What payment methods do you accept?", 
        "answer": "We accept cash, credit/debit cards (Visa, Mastercard, American Express), bank transfers, and payment through major insurance providers (Sanitas, Adeslas, Mapfre, DKV, Asisa). Payment plans are available for treatments exceeding €500." 
    }, 
    "FAQ005": { 
        "category": "Payments", 
        "question": "How can I check my balance or pay outstanding bills?", 
        "answer": "You can view and pay your balance through our patient portal at medicare-plus.com/billing, by phone at +34 911 234 568 (billing department), or in person at our reception desk. We send monthly statements via email for any outstanding balances." 
    }, 
    "FAQ006": { 
        "category": "Insurance", 
        "question": "Which insurance providers do you work with?", 
        "answer": "We work with Sanitas, Adeslas, Mapfre, DKV, and Asisa. Coverage varies by plan. Please bring your insurance card to every visit. We recommend calling your insurance provider to verify coverage before specialized procedures." 
    }, 
    "FAQ007": { 
        "category": "Services", 
        "question": "What medical specialties are available at the clinic?", 
        "answer": "MediCare Plus offers: General Medicine, Pediatrics, Gynecology, Dermatology, Cardiology, Traumatology, Psychology, and Nutrition. Some specialists are only available on specific days. Contact reception for specialist schedules." 
    }, 
    "FAQ008": { 
        "category": "Services", 
        "question": "Do you offer telemedicine consultations?", 
        "answer": "Yes, we offer video consultations for General Medicine, Psychology, Nutrition, and follow-up appointments. Telemedicine is available for Premium members at no extra cost. Basic members pay €15 per video consultation. Book through our portal or call reception." 
    }, 
    "FAQ009": { 
        "category": "Hours & Location", 
        "question": "What are the clinic's operating hours?", 
        "answer": "Regular hours: Monday to Friday 8:00-20:00, Saturday 9:00-14:00. Closed on Sundays and public holidays. Extended hours available by appointment for Premium members." 
    }, 
    "FAQ010": { 
        "category": "Hours & Location", 
        "question": "Where is the clinic located?", 
        "answer": "We are located at Calle Serrano 125, 28006 Madrid. Nearest metro:Núñez de Balboa (Line 5, 9). Paid parking available at Parking Serrano (2-minute walk). Wheelchair accessible entrance on Calle Claudio Coello." 
    }, 
    "FAQ011": { 
        "category": "Emergencies", 
        "question": "Do you handle medical emergencies?", 
        "answer": "MediCare Plus is NOT an emergency facility. For medical emergencies, call 112 or go to the nearest hospital emergency room. For urgent but non-emergency same-day care, Premium members can request urgent appointments subject to availability." 
    }, 
    "FAQ012": { 
        "category": "Membership", 
        "question": "What is the difference between Basic and Premium membership?", 
        "answer": "Basic membership: Standard appointment scheduling, access to all specialists, email support. Premium membership (€29/month): Priority scheduling, 12hour cancellation policy, free telemedicine, extended hours access, dedicated phone support line, and 10% discount on non-covered services." 
    }, 
    "FAQ013": { 
        "category": "Medical Records", 
        "question": "How can I access my medical records?", 
        "answer": "Access your records through our patient portal at medicareplus.com/records. You can also request printed copies at reception (€5 processing fee, ready within 48 hours). For records to be sent to another provider, submit a signed authorization form." 
    }, 
    "FAQ014": { 
        "category": "Prescriptions", 
        "question": "How do I request prescription refills?", 
        "answer": "Prescription refills can be requested through the patient portal or by calling your doctor's office directly. Please allow 48-72 hours for processing. Controlled substances require an in-person appointment." 
    }, 
    "FAQ015": { 
        "category": "COVID-19", 
        "question": "What COVID-19 protocols are in place?", 
"answer": "Masks are optional but recommended in waiting areas. Hand sanitizer stations are available throughout the clinic. If you have COVID-19 symptoms, please call before your visit to arrange appropriate care. Telemedicine is recommended for patients with respiratory symptoms." 
} 
}

## Intent Detection (Layer 1)

In [ ]:
class Intent(Enum):
    EMERGENCY = "emergency"
    PERSONAL_DATA = "personal_data"
    FAQ = "faq"
    UNCLEAR = "unclear"

In [ ]:
def detect_intent_llm(message: str, conversation_history: List[Dict[str, str]]) -> Intent:

    history_context = ""
    if conversation_history:
        history_context = "Previous conversation:\n"
        for turn in conversation_history[-3:]:
            role = "User" if turn["role"] == "user" else "Assistant"
            history_context += f"{role}: {turn['content']}\n"

    prompt = f"""
    You are an intent classifier for MediCare Plus medical clinic.
    
    {history_context}
    
    Classify the user's current message into exactly one of these categories:
    
    1. EMERGENCY: Medical emergencies, urgent symptoms, life-threatening situations that require immediate 911/112.
    2. PERSONAL_DATA: Questions about the user's personal information (my balance, my appointment, my doctor, my insurance, etc.)
    3. FAQ: General questions about clinic operations, services, policies, hours, location, etc.
    4. UNCLEAR: Anything else, including questions about specific medical procedures we don't offer
    
    Current message: "{message}"
    
    Think step by step:
    1. Is this a medical emergency? If yes, EMERGENCY
    2. Is this about the user's personal medical data/account? If yes, PERSONAL_DATA  
    3. Is this a general question about clinic operations? If yes, FAQ
    4. Otherwise, UNCLEAR
    
    Respond with ONLY the category name: EMERGENCY, PERSONAL_DATA, FAQ, or UNCLEAR
    """

    try:
        response = llm.invoke(prompt)
        intent_str = response.content.strip().upper()

        
        if "EMERGENCY" in intent_str:
            return Intent.EMERGENCY
        elif "PERSONAL_DATA" in intent_str:
            return Intent.PERSONAL_DATA
        elif "FAQ" in intent_str:
            return Intent.FAQ
        else:
            return Intent.UNCLEAR

    except Exception as e:
        print(f"LLM intent detection failed: {e}")
        return detect_intent_rules(message, conversation_history)


def detect_intent_rules(message: str, conversation_history: List[Dict[str, str]]) -> Intent: #Rule based intent detection as fallback

    message_lower = message.lower()

    
    emergency_keywords = [
        "emergency", "urgent", "severe chest pain", "heart attack", "stroke",
        "bleeding heavily", "unconscious", "can't breathe", "not breathing",
        "dying", "ambulance", "severe pain", "chest pain", "having severe",
        "call 112", "help me", "critical", "life threatening", "911"
    ]

    if any(keyword in message_lower for keyword in emergency_keywords):
        return Intent.EMERGENCY


    personal_indicators = [
        "my ", "do i owe", "my balance", "my appointment", "who is my",
        "what is my", "when is my", "my doctor", "my insurance", "my records",
        "my prescription", "i owe", "my upcoming", "my primary", "my next visit",
        "my allergies", "my last visit", "my membership", "my bill"
    ]

    if any(indicator in message_lower for indicator in personal_indicators):
        return Intent.PERSONAL_DATA


    faq_keywords = set()
    for faq_id, faq in CLINIC_FAQS.items():
        question_words = set(faq["question"].lower().split())
        faq_keywords.update(question_words)

    message_words = set(message_lower.split())
    stop_words = set(stopwords.words('english'))
    relevant_words = message_words.intersection(faq_keywords) - stop_words

    if len(relevant_words) >= 2:
        return Intent.FAQ

    return Intent.UNCLEAR

# Set detect_intent to use the LLM version by default
detect_intent = detect_intent_llm

## RAG Logic (Layer 2)

In [ ]:
def retrieve_faq_with_llm(message: str) -> Optional[Dict[str, Any]]:

    faq_list = []
    for faq_id, faq in CLINIC_FAQS.items():
        faq_list.append(f"[{faq_id}] {faq['question']} - Category: {faq['category']}")
    
    faq_text = "\n".join(faq_list)
    
    prompt = f"""
    You are a FAQ matcher for MediCare Plus clinic.
    
    Here are all the available FAQs:
    {faq_text}
    
    User's question: "{message}"
    
    Your task:
    1. Find the FAQ that best answers the user's question
    2. If no FAQ is relevant, say "NONE"
    3. If a FAQ is relevant, respond with JUST the FAQ ID (e.g., FAQ001)
    
    Think step by step:
    - What is the user asking about?
    - Which FAQ(s) address this topic?
    - Choose the most specific and relevant FAQ
    
    Remember: Only respond with a FAQ ID or "NONE"
    """
    
    try:
        response = llm.invoke(prompt)
        faq_id = response.content.strip()
        
        if faq_id in CLINIC_FAQS:
            faq = CLINIC_FAQS[faq_id]
            return {
                "faq_id": faq_id,
                "question": faq["question"],
                "answer": faq["answer"],
                "category": faq["category"],
                "score": 10  
            }
        elif faq_id != "NONE":
            
            for id in CLINIC_FAQS.keys():
                if id in response.content:
                    faq = CLINIC_FAQS[id]
                    return {
                        "faq_id": id,
                        "question": faq["question"],
                        "answer": faq["answer"],
                        "category": faq["category"],
                        "score": 8
                    }
    
    except Exception as e:
        print(f"LLM FAQ retrieval failed: {e}")
        
        return retrieve_faq_keyword(message)
    
    return None

def retrieve_faq_keyword(message: str) -> Optional[Dict[str, Any]]:

    message_lower = message.lower()
    best_match = None
    best_score = 0
    
    for faq_id, faq in CLINIC_FAQS.items():
        question = faq["question"].lower()
        answer = faq["answer"].lower()
        category = faq["category"].lower()
        
        score = 0
        
        # Check question words
        question_words = set(question.split())
        message_words = set(message_lower.split())
        common_words = question_words.intersection(message_words)
        score += len(common_words) * 2
        
        # Check if key terms are in the question
        if "specialty" in message_lower and "specialty" in question:
            score += 5
        if "appointment" in message_lower and "appointment" in question:
            score += 5
        if "payment" in message_lower and "payment" in question:
            score += 5
        if "insurance" in message_lower and "insurance" in question:
            score += 5
        if "hour" in message_lower and "hour" in question:
            score += 5
        if "location" in message_lower and "location" in question:
            score += 5
        
        if score > best_score:
            best_score = score
            best_match = {
                "faq_id": faq_id,
                "question": faq["question"],
                "answer": faq["answer"],
                "category": faq["category"],
                "score": score
            }
    
    if best_match and best_score >= 3:
        return best_match
    return None



In [ ]:
def obtain_patient_data_llm(user_id: str, message: str) -> Optional[str]:

    if user_id not in PATIENTS_DB:
        return None
    
    patient = PATIENTS_DB[user_id]
    first_name = patient["name"].split()[0]
    
    
    doctor_specialties = {}
    for appt in patient.get("upcoming_appointments", []):
        doctor = appt.get("doctor")
        specialty = appt.get("specialty")
        if doctor and specialty:
            doctor_specialties[doctor] = specialty
    

    primary_doctor = patient['primary_doctor']
    primary_doctor_specialty = doctor_specialties.get(primary_doctor, "Not specified in records")
    

    patient_info = f"""
    Patient Information for {patient['name']}:
    
    Personal Details:
    - Email: {patient['email']}
    - Phone: {patient['phone']}
    - Date of Birth: {patient['date_of_birth']}
    - Membership Type: {patient['membership_type']}
    - Insurance Provider: {patient['insurance_provider']}
    - Primary Doctor: {primary_doctor} (Specialty: {primary_doctor_specialty})
    
    Medical Information:
    - Allergies: {', '.join(patient['allergies']) if patient['allergies'] else 'None'}
    - Last Visit: {patient['last_visit']}
    - Balance Due: €{patient['balance_due']:.2f}
    
    Upcoming Appointments:"""
    
    if patient['upcoming_appointments']:
        for i, appt in enumerate(patient['upcoming_appointments'], 1):
            patient_info += f"\n{i}. Date: {appt['date']}, Time: {appt['time']}, "
            patient_info += f"Doctor: {appt['doctor']}, Specialty: {appt['specialty']}, Status: {appt['status']}"
    else:
        patient_info += "\nNo upcoming appointments."
    

    if doctor_specialties:
        patient_info += "\n\nDoctor-Specialty Reference:"
        for doc, spec in doctor_specialties.items():
            patient_info += f"\n- {doc}: {spec}"
    
    prompt = f"""
    You are a medical assistant at MediCare Plus clinic. 
    You have access to this patient's information:
    
    {patient_info}
    
    Patient query: "{message}"
    
    Your task:
    1. Answer the patient's question based ONLY on the information above
    2. If the information is not available, say so
    3. Keep the answer concise and friendly
    4. Address the patient by their first name: {first_name}
    5. Include the specific data field(s) you're referencing
    6. If they ask to cancel or reschedule an appointment, provide the date, time, doctor, and specialty of the relevant appointment(s) and instruct them to call reception or use the online portal to make changes.
    
    Important: Do NOT make up any information. Only use what's provided.
    Important: If asked about primary doctor's specialty, use the specialty shown next to their name.
    
    Format your response like this:
    [Answer: Your answer here] [Source: Patient Data - field_name]
    
    For example:
    [Answer: Hi Maria, your next appointment is on 2024-02-15 at 10:30 with Dr. Fernandez.] [Source: Patient Data - upcoming_appointments]
    [Answer: Hi Maria, Dr. Fernandez specializes in General Medicine.] [Source: Patient Data - primary_doctor_specialty]
    
    Now respond to: "{message}"
    """
    
    try:
        response = llm.invoke(prompt)
        answer = response.content.strip()
        

        if "[Answer:" in answer and "[Source:" in answer:
            return answer
        else:

            return f"[Answer: {answer}] [Source: Patient Data - Comprehensive]"
            
    except Exception as e:
        print(f"LLM patient data retrieval failed: {e}")

        return obtain_patient_data_rules(user_id, message)

def obtain_patient_data_rules(user_id: str, message: str) -> Optional[str]:

    if user_id not in PATIENTS_DB:
        return None

    patient = PATIENTS_DB[user_id]
    first_name = patient["name"].split()[0]
    message_lower = message.lower()


    doctor_specialties = {}
    for appt in patient.get("upcoming_appointments", []):
        doctor = appt.get("doctor")
        specialty = appt.get("specialty")
        if doctor and specialty:
            doctor_specialties[doctor] = specialty



    if any(word in message_lower for word in ["doctor", "primary", "physician", "dr"]):
        primary_doctor = patient['primary_doctor']
        specialty = doctor_specialties.get(primary_doctor, "specialty not listed")

        if any(word in message_lower for word in ["specialty", "specialist", "field", "area"]):
            return f"[Answer: Hi {first_name}, Dr. {primary_doctor.split()[-1]} specializes in {specialty}.] [Source: Patient Data - primary_doctor_specialty]"
        
        return f"[Answer: Hi {first_name}, your primary doctor is {primary_doctor} ({specialty}).] [Source: Patient Data - primary_doctor]"


## Contextual Memory (Layer 3)

In [ ]:
def ref_resolver(message: str, conversation_history: List[Dict[str, str]]) -> Tuple[str, Optional[str]]:

    if not conversation_history:
        return message, None


    mentioned_doctors = []
    mentioned_specialties = []

    doctor_pattern = r"Dr\. \w+"
    specialties = [
        "General Medicine", "Pediatrics", "Gynecology", "Dermatology",
        "Cardiology", "Traumatology", "Psychology", "Nutrition"
    ]

    for turn in conversation_history:
        content = turn.get("content", "")


        doctors = re.findall(doctor_pattern, content)
        mentioned_doctors.extend(doctors)


        for specialty in specialties:
            if specialty.lower() in content.lower():
                mentioned_specialties.append(specialty)

    last_doctor = mentioned_doctors[-1] if mentioned_doctors else None


    resolved = message

    if last_doctor:
        
        patterns = [
            (r'\bhis\b(?=\s+(specialty|doctor|appointment|availability))', f"{last_doctor}'s"),
            (r'\bher\b(?=\s+(specialty|doctor|appointment|availability))', f"{last_doctor}'s"),
            (r'\btheir\b(?=\s+(specialty|doctor|appointment|availability))', f"{last_doctor}'s"),
            (r'\bthat doctor\b', last_doctor),
            (r'\bthe same doctor\b', last_doctor),
            (r'\bit\b(?=\s+(specialty|available))', f"{last_doctor}'s specialty")
        ]

        for pattern, replacement in patterns:
            resolved = re.sub(pattern, replacement, resolved, flags=re.IGNORECASE)

    if mentioned_specialties:
        last_specialty = mentioned_specialties[-1]
        resolved = re.sub(r'\bthat specialty\b', last_specialty, resolved, flags=re.IGNORECASE)

    return resolved, last_doctor

## Escalation Checks (Layer 4)

In [ ]:
def llm_escalation_check(message: str, intent: Intent, rag_result: Optional[Any]) -> Tuple[bool, str]:

    message_lower = message.lower()
    

    if intent == Intent.EMERGENCY:
        return True, "emergency"
    

    if intent == Intent.UNCLEAR or (intent == Intent.FAQ and not rag_result):
        prompt = f"""
        You are a medical chatbot assistant. Determine if this user query should be escalated to a human agent.
        
        User query: "{message}"
        
        Reasons to escalate:
        1. Asking about complex medical procedures we don't offer (transplants, major surgery, etc.)
        2. Complex medical advice or diagnosis
        3. Questions about specific medications or treatments we don't cover
        4. Complaints about staff or service quality
        5. Billing disputes or complex financial questions
        6. Legal or insurance disputes
        
        Reasons NOT to escalate:
        1. General questions about clinic operations (hours, location, services)
        2. Simple appointment or billing questions
        3. Questions we have FAQs for
        4. Simple information requests
        
        Should this be escalated to a human agent? Answer YES or NO.
        """
        
        try:
            response = llm.invoke(prompt)
            decision = response.content.strip().upper()
            
            if "YES" in decision:

                if any(word in message_lower for word in ["transplant", "surgery", "chemotherapy", "radiation"]):
                    return True, "complex_medical"
                elif any(word in message_lower for word in ["angry", "frustrated", "complaint", "sue"]):
                    return True, "negative_sentiment"
                else:
                    return True, "unanswerable"
        except Exception as e:
            print(f"LLM escalation check failed: {e}")
    

    negative_keywords = [
        "angry", "frustrated", "furious", "unacceptable", "ridiculous",
        "terrible", "worst", "hate", "useless", "incompetent"
    ]
    
    if any(keyword in message_lower for keyword in negative_keywords):
        return True, "negative_sentiment"
    

    procedure_keywords = ["transplant", "surgery", "chemotherapy", "dialysis", "bypass"]
    if any(keyword in message_lower for keyword in procedure_keywords):
        return True, "complex_medical"
    

    urgency_keywords = ["asap", "urgent", "immediate", "right now", "today"]
    if any(keyword in message_lower for keyword in urgency_keywords):
        return True, "urgency"
    
    return False, "none"

## Output Formatting (Layer 5)

In [ ]:
def format_response(content: str, source_type: str, source_id: str) -> str:

    if source_type == "faq":
        return f"{content} [Source: {source_id}]"
    elif source_type == "patient_data":
        return f"{content} [Source: {source_id}]"
    else:
        return content

def format_escalation_message(reason: str) -> str:

    messages = {
        "emergency": "  I understand this is an emergency. I'm immediately transferring you to a human agent. Please stay on the line. Transferring to agent...",
        "negative_sentiment": "I'm sorry to hear you're frustrated. Let me connect you with a human agent who can better assist you. Transferring to agent...",
        "unanswerable": "I'm sorry, I cannot find information related to your query in our database. Would you like to speak to a human agent who can help you further?",
        "urgency": "I understand this is urgent. Let me transfer you to a human agent who can provide immediate assistance. Transferring to agent..."
    }
    return messages.get(reason, "Transferring to agent...")


## Main function with LangSmith tracing

In [ ]:
def generate_response(
    user_id: str,
    current_message: str,
    conversation_history: List[Dict[str, str]]
) -> Dict[str, Any]:
    """     
    Inputs: user_id: Patient ID (e.g., "P001") 
    current_message: The latest text from the user 
    conversation_history: List of past turns [{"role": "user", "content": "..."}, {"role": "bot", "content": "..."}] 
     
    Returns: 
        { 
            "response": str, 
            "escalate_to_human": bool 
        } 
    """ 
    
    
    
    with tracing_v2_enabled(project_name="medicare-chatbot-chat-widget"):
        
        resolved_message, last_doctor = ref_resolver(current_message, conversation_history)
        
        
        intent = detect_intent(resolved_message, conversation_history)
        
        
        rag_result = None
        source_type = None
        source_id = None
        
        if intent == Intent.EMERGENCY:
            pass
            
        elif intent == Intent.PERSONAL_DATA:

            patient_response = obtain_patient_data_llm(user_id, resolved_message)
            if patient_response:
                rag_result = patient_response
                source_type = "patient_data"
                if "[Source:" in patient_response:
                    source_id = patient_response.split("[Source: ")[1].split("]")[0]
        
        elif intent == Intent.FAQ:

            faq_match = retrieve_faq_with_llm(resolved_message)
            if faq_match:
                rag_result = faq_match["answer"]
                source_type = "faq"
                source_id = faq_match["faq_id"]
            else:
                faq_match = retrieve_faq_keyword(resolved_message)
                if faq_match:
                    rag_result = faq_match["answer"]
                    source_type = "faq"
                    source_id = faq_match["faq_id"]
        

        should_escalate, escalation_reason = llm_escalation_check(
            current_message, intent, rag_result
        )
        

        if should_escalate:
            response = format_escalation_message(escalation_reason)
        else:
            if rag_result:

                if rag_result.startswith("[Answer:") and rag_result.endswith("]"):
                    answer_part = rag_result.split("[Answer: ")[1].split("] [Source:")[0]
                    response = answer_part
                else:
                    response = rag_result
            else:
                response = "I'm sorry, I cannot find information related to your query. Would you like to speak to a human agent?"
                should_escalate = True
        
        return {
            "response": response,
            "escalate_to_human": should_escalate
        }


## Test all scenarios with LangSmith tracing

In [ ]:
# print("\n" + "="*70)
# print("TESTING ENHANCED PATIENT DATA RETRIEVAL WITH LLM")
# print("="*70)

# test_queries = [
#     ("What's my next appointment?", "Should show appointment details"),
#     ("Who is my primary doctor?", "Should show primary doctor"),
#     ("Do I owe any money?", "Should show balance"),
#     ("What are my allergies?", "Should show allergies"),
#     ("What's my insurance?", "Should show insurance provider"),
#     ("When was my last visit?", "Should show last visit date"),
#     ("Is Dr. Fernández available this week?", "Complex query - should use LLM"),
#     ("Can I see a dermatologist soon?", "Complex query - should use LLM"),
# ]

# for query, expected in test_queries:
#     print(f"\nQuery: {query}")
#     result = generate_response("P001", query, [])
    
#     if result['escalate_to_human']:
#         print(f"Result: ESCALATING - {result['response'][:80]}...")
#     else:
#         print(f"Result: {result['response']}")
    
#     print("-" * 50)

### Chatbot UI widget for further tests

In [ ]:

user_id = widgets.Dropdown(
    options=['P001', 'P002', 'P003'],
    value='P001',
    description='Patient:'
)

message_input = widgets.Text(
    placeholder='Type your question here...',
    description='Question:',
    layout=widgets.Layout(width='80%')
)

send_button = widgets.Button(description='Send', button_style='success')
clear_button = widgets.Button(description='Clear Chat', button_style='danger')

output = widgets.Output(layout={'border': '1px solid black', 'height': '300px', 'width': '100%',})
conversation_history = []

def wrap_text_simple(text, width=100):
    """Simple text wrapping that breaks lines at specified width"""
    wrapped = []
    for line in text.split('\n'):
        if len(line) <= width:
            wrapped.append(line)
        else:

            for i in range(0, len(line), width):
                wrapped.append(line[i:i+width])
    return '\n'.join(wrapped)

def on_send_clicked(b = None):
    message = message_input.value
    pid = user_id.value
    
    if message:

        with output:
            print(f"User [{pid}]: {message}")
        

        result = generate_response(pid, message, conversation_history)
        

        wrapped_response = wrap_text_simple(result['response'], 100)
        print(f"")
        
        with output:
            if result['escalate_to_human']:
                print(f" Medicare Plus Bot: \n {wrapped_response}")
            else:
                print(f"Medicare Plus Bot: \n {wrapped_response}")

        
             
            print("-" * 40)
        

        conversation_history.append({"role": "user", "content": message})
        conversation_history.append({"role": "bot", "content": result['response']})
        

        message_input.value = ''

def on_clearchat_clicked(b):
    output.clear_output()
    conversation_history.clear()
    with output:
        print("Chat cleared!")

send_button.on_click(on_send_clicked)
message_input.on_submit(lambda x: on_send_clicked())
clear_button.on_click(on_clearchat_clicked)


display(widgets.VBox([
    widgets.HTML("<h3>MediCare Plus Chatbot</h3>"),
    user_id,
    widgets.HBox([message_input, send_button, clear_button]),
    output
]))